In [1]:
import azure.ai.ml
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment, AmlCompute, Data, Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
import mlflow

c:\Users\Dell\PycharmProjects\Azure ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#--------------------------------------------------------------------------------------
# Set up the Workspace
#--------------------------------------------------------------------------------------
credential = InteractiveBrowserCredential(
    tenant_id = "a8fdbb05-558e-4cbc-81a7-375d6e278fa3"
)

ml_client  = MLClient(
    credential=credential,
    subscription_id="e2e1c6ce-6a9b-42b5-a067-321c488b1949",
    resource_group_name="Joshua_resource",
    workspace_name="Joshua-Space"
)

mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
c:\Users\Dell\PycharmProjects\Azure ML\.venv\Lib\site-packages\msal\oauth2cli\oauth2.py:408: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


In [3]:
# -----------------------------------------------------------------------------------
#Setting up computing componenet
# --------------------------------------------------=--------------------------------
ml_client.compute.begin_delete(name="class-deep-learning").wait()
print("Deleted compute target 'class-deep-learning' if it existed.")

try:
    compute_name = "class-deep-learning"
    comp = ml_client.compute.get(name=compute_name)
    print(f"Found existing compute target: {comp.name}")
except Exception :
    comp = AmlCompute(
        name=compute_name,
        size="Standard_F4s_v2",
        min_instances=0,
        max_instances=4
    )
    ml_client.compute.begin_create_or_update(comp).wait()


for comp in ml_client.compute.list():
    print(comp.name)

# -----------------------------------------------------------------------------------
# Setting up Computing Instance
# -----------------------------------------------------------------------------------

instance = ml_client.compute.get(name="Klint-Instance")
print("Current Instance state: ", instance.state)

if instance.state != "Running":
    print(f"Instance about to start running")
    ml_client.compute.begin_start(name="Klint-Instance").wait()

instance = ml_client.compute.get(name="Klint-Instance")
print("After Instance state: ", instance.state)

Deleted compute target 'class-deep-learning' if it existed.
Klint-Instance
klint-cluster
class-cluster
class-deep-learning
Current Instance state:  Stopped
Instance about to start running
After Instance state:  Running


In [4]:
for data_name in ml_client.data.list():
    print(data_name.name)
    
data_name = "Cat-Dog"

# try:
#     dataExist = ml_client.data.get(name=data_name)
#     print(f"{dataExist.name} Exist")
# except Exception: 
#     Image_data = Data(
#     name='Cat-Dog',
#     type= AssetTypes.URI_FOLDER ,
#     path= "./cat_dog",
#     description="Dogs and cat data for classification"
#     )

#     ml_client.data.create_or_update(Image_data)
    
#   print(f"Data asset '{data_name}' created successfully")

dataset = ml_client.data.get(name=data_name, label="latest")
print(f"Dataset '{dataset.name}' with path '{dataset.path}' retrieved successfully")


Chicago
new_chicago
MD-Design-Train_Model-Trained_model-d76755ea
diabetes
dataset
validation_0
Cat-Dog
Dataset 'Cat-Dog' with path 'azureml://subscriptions/e2e1c6ce-6a9b-42b5-a067-321c488b1949/resourcegroups/Joshua_resource/workspaces/Joshua-Space/datastores/workspaceblobstore/paths/LocalUpload/1f3ba16437b1ca3a6c6899f0115b77834a105b9a96fa9d8012cec3215c1ea2cc/cat_dog/' retrieved successfully


In [5]:
dataset = ml_client.data.get(name="Cat-Dog", label="latest")
print(dataset.path) 

azureml://subscriptions/e2e1c6ce-6a9b-42b5-a067-321c488b1949/resourcegroups/Joshua_resource/workspaces/Joshua-Space/datastores/workspaceblobstore/paths/LocalUpload/1f3ba16437b1ca3a6c6899f0115b77834a105b9a96fa9d8012cec3215c1ea2cc/cat_dog/


In [6]:
#------------------------------------------------------------------------------------
# Setting up Envionment
#------------------------------------------------------------------------------------

env_name = "class-deep-learning-env"

env  = Environment(
    name=env_name,
    conda_file="./conda_dependencies.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    description="Environment for deep learning class"
)

env =ml_client.environments.create_or_update(env)
print(f"Environment '{env.name}' created successfully")

Environment 'class-deep-learning-env' created successfully


In [9]:
#------------------------------------------------------------------------------------
# Setting up Job
#------------------------------------------------------------------------------------   

job = command(
    code="./",
    description="Job for deep learning class",
    command="python AzTrain.py --input ${{inputs.data}}",
    inputs={
        "data": Input(type=AssetTypes.URI_FOLDER, path=dataset.path)
    },
    environment=f"{env_name}:{env.version}",
    compute="class-deep-learning",
    experiment_name="class-deep-learning-experiment",
    display_name="Deep Learning Training Job"
)

job = ml_client.jobs.create_or_update(job)
print(f"Job '{job.name}' created successfully with status: {job.status}")



Uploading src (0.78 MBs): 100%|##########| 778249/778249 [00:05<00:00, 137863.71it/s]




Job 'mighty_head_nqv5dlhlkm' created successfully with status: Starting
